# Publication-oriented X-SHOOTER UVB scaffold

This notebook is an expert workflow scaffold, not the first example a new user should run. It mirrors `publication_quality_xshooter_uvb.py` and starts with audit-only checks: explicit Balmer windows, documented masks, ordinary fit-readiness, and stricter publication-readiness. The expensive PHOENIX baseline fit is opt-in below.

A passing publication-readiness gate is not by itself a publication-quality result. It only says the input assumptions are documented enough to proceed to systematic variants, per-line checks, injection/recovery, and final uncertainty accounting.

In [ ]:
from pathlib import Path
import json
import sys

# Locate the source checkout whether Jupyter starts in the repo root or examples/.
repo = Path.cwd()
if not (repo / "Spyctres").is_dir() and (repo.parent / "Spyctres").is_dir():
    repo = repo.parent
sys.path.insert(0, str(repo))
sys.path.insert(0, str(repo / "examples"))

from publication_quality_xshooter_uvb import main

## 1. Run the safe audit-only scaffold

This does not require PHOENIX. It prepares the bundled X-SHOOTER UVB spectrum as explicit Balmer-window segments and writes a JSON checkpoint under `/tmp`.

In [ ]:
audit_json = Path("/tmp/spyctres_publication_xshooter_uvb_notebook.json")
main([
    "--output-json", str(audit_json),
    "--force",
])

In [ ]:
payload = json.loads(audit_json.read_text())
{
    "status": payload["status"],
    "publication_ready": payload["publication_readiness"]["publication_ready"],
    "blockers": payload["publication_readiness"]["blockers"],
    "n_fit_candidate": payload["ordinary_readiness"]["n_fit_candidate"],
}

## 2. Inspect the Balmer-core mask sensitivity grid

The baseline core mask is not assumed to be correct. The script records an audit-only grid so we can see how much data each mask width removes before deciding which widths deserve expensive PHOENIX refits.

In [ ]:
[
    {
        "core_mask_halfwidth_A": item["core_mask_halfwidth_A"],
        "n_fit_candidate": item["n_fit_candidate"],
        "rejected_inside_fraction": item["rejected_inside_fit_window_fraction"],
        "publication_blockers": item["publication_blockers"],
    }
    for item in payload["core_mask_sensitivity"]
]

## 3. Optional PHOENIX baseline fit

Set `RUN_BASELINE_FIT = True` only after the PHOENIX directory is configured. This is still a baseline scaffold: it does not replace systematic variants, per-line checks, or injection/recovery validation.

In [ ]:
RUN_BASELINE_FIT = False

if RUN_BASELINE_FIT:
    fit_json = Path("/tmp/spyctres_publication_xshooter_uvb_fit_notebook.json")
    fit_plot = Path("/tmp/spyctres_publication_xshooter_uvb_fit_notebook.png")
    main([
        "--run-baseline-fit",
        "--output-json", str(fit_json),
        "--output-plot", str(fit_plot),
        "--force",
    ])
    fit_payload = json.loads(fit_json.read_text())
    {
        "status": fit_payload["status"],
        "teff": fit_payload["baseline_fit"].get("teff"),
        "logg": fit_payload["baseline_fit"].get("logg"),
        "feh": fit_payload["baseline_fit"].get("feh"),
        "rv_kms": fit_payload["baseline_fit"].get("rv_kms"),
        "chi2_red": fit_payload["baseline_fit"].get("chi2_red"),
        "plot": str(fit_plot),
    }